# Exploring the zebra null distributions — SLAP2

The SLAP2 counterpart of `explore_null_results.ipynb` (ecephys). Same statistic,
same null, same Gumbel tail, same BH correction; this notebook adapts the chapters
that turn the null files into a significance call — **running the check**, **4**
(the per-unit statistic), **5** (individual units) and **6** (FDR across the
session) — to what the SLAP2 results look like. Everything else is kept to the
minimum those chapters need.

What is different from ephys, and matters for reading the numbers:

* **There are no trials.** SLAP2 images in bouts with blanking in between, so a
  single repeat may keep too few frames to correlate against. The pipeline pools all
  repeats into one design matrix and the null does the same, so the path has no
  `trial_<i>` level (`pooled_repeats` in the attributes). Rolls are along the
  retained-frame axis, and the pooling means one roll of about one repeat length
  re-pairs each response with nearly the frame it really saw — a *conservative* flaw
  in the null, discussed in `compute_null_distribution_slap2.py`.
* **The "probes" are (DMD, channel) pairs.** Two DMDs x two channels, each its own
  file: **green** is iGluSnFR4f and resolves single movie frames, **red** is RCaMP3
  whose calcium kinetics blur the response across frames. Each has its own delay and
  duration grid in the pipeline, copied into the null.
* **The signal here is dF/F** (`dff`); the pipeline also has an `events` variant
  with no null files yet.
* **It is a small session** — 250 ROIs against ~3 100 units in ecephys — so the BH
  line is much steeper per rank and every count below is small. Read the tables as
  tables, not as rates.
* **The null grid is the pipeline grid** (6 delays x 3 durations), so no restriction
  to a sub-grid is needed, and there is **no `validate/` run**: the check compares
  the null's lag 0 against the *whole-grid* maximum in the optimized results.
* **`K = 300` surrogates**, so the counting floor is 1/301 = 0.0033 (ephys: 0.0099).

In [ ]:
import sys
from pathlib import Path

sys.path.append('..')
sys.path.append('../scripts')

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NULL_ROOT = Path('../results/zebra/nullresults/slap2')
OPT_ROOT = Path('../results/zebra/optimized/slap2')
SESSION = 'sub-829704_ses-829704-2025-12-18-10-57-36_image+ophys'
SIGNAL = 'dff'
# The optimized dF/F results sit directly under the session directory; the events
# variant lives in an `events/` subdirectory with the SAME file names.
OPT_DIR = OPT_ROOT / SESSION / ('' if SIGNAL == 'dff' else SIGNAL)

# One phase is one complete analysis of the session (no trials: repeats are pooled).
PHASE = '0'

WALK = dict(dmd='DMD1', channel='green', phase='0')   # the file walked through below

ACTIVITY = 'dF/F samples'     # what n_active counts, for labels
GROUPS = 'DMD x channel files'
SEED = 7                      # the sampled units of chapter 5 are drawn with this
Q_LEVEL = 0.05                # FDR level used throughout

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', None)

## 1. How the tree is organised, and what one file contains

One HDF5 file per **(session, signal, DMD, channel, phase, library)**; the path is
the key:

```
nullresults/slap2/<session>/<signal>/<dmd>/<channel>/phase_<p>/lib_<id>/null__lib_<id>.h5
```

`family_max` is the whole result, with shape `(n_units, n_families, 1 + K)`: lag
column `0` is the **observed** per-family maximum and columns `1:` are the `K`
circularly rolled surrogates. A family is one (sigma, frequency) combination of the
Gabor library — 78 of them.

In [ ]:
def null_index(root=NULL_ROOT):
    """One row per null file, with the path decomposed into its keys."""
    rows = []
    for f in sorted(Path(root).rglob('null__lib_*.h5')):
        session, signal, dmd, channel, phase, lib = f.relative_to(root).parts[:6]
        rows.append({'session': session, 'signal': signal, 'dmd': dmd, 'channel': channel,
                     'phase': phase.split('_')[1], 'lib': lib.split('_')[1],
                     'MB': round(f.stat().st_size / 1e6, 1), 'path': f})
    return pd.DataFrame(rows)


def pick(index, **keys):
    """The single path in `index` matching every key=value given."""
    mask = np.ones(len(index), dtype=bool)
    for k, v in keys.items():
        mask &= (index[k] == v).to_numpy()
    if mask.sum() != 1:
        raise LookupError(f'{mask.sum()} files match {keys}, expected exactly 1')
    return index.loc[mask, 'path'].iloc[0]


def load_null(path):
    """Everything in one null file, as a dict."""
    with h5py.File(path) as hf:
        out = {k: v for k, v in hf.attrs.items()}
        out['family_max'] = hf['family_max'][:]
        out['unit_ids'] = hf['unit_ids'][:].astype(str)
        out['n_spikes'] = hf['n_spikes'][:]          # activity count, see attrs
        out['lags'] = hf['lags'][:]
        out['lag_seconds'] = hf['lag_seconds'][:]
        out['path'] = Path(path)
    return out


idx = null_index()
print(f'{len(idx)} null files, {idx.MB.sum():.0f} MB\n')

walk_path = pick(idx, session=SESSION, signal=SIGNAL, **WALK)
nl = load_null(walk_path)
n_units, n_families, n_lags = nl['family_max'].shape
print(walk_path.relative_to(NULL_ROOT), '\n')
print(f"family_max    {nl['family_max'].shape} = (units, families, 1 observed + {n_lags - 1} surrogates)")
print(f"sweep         {len(nl['sweep_delays'])} delays x {len(nl['sweep_durations'])} durations"
      f" = {len(nl['grid_delays'])} points; {nl['n_row_selections']} distinct row selections")
print(f"lags          {nl['lags'][:4]} ... {nl['lags'][-2:]} {nl['roll_axis']}"
      f"  (>= {nl['lag_seconds'][1]:.1f} s smallest roll, a lower bound; min_lag_sec={nl['min_lag_sec']})")
print(f"retained frames per design matrix: {nl['n_timepoints_min']}-{nl['n_timepoints_max']} "
      f"of {nl['n_frames']} x repeats; coverage >= {nl['min_coverage']}")
print(f"units         {n_units} with >= {nl['min_active_samples']} {nl['activity_measure']}; "
      f"n_active {nl['n_spikes'].min()}-{nl['n_spikes'].max()}")
observed = nl['family_max'][:, :, 0]        # (units, families)
surrogate = nl['family_max'][:, :, 1:]      # (units, families, K)
print(f"observed  mean {observed.mean():.4f}  max {observed.max():.4f}")
print(f"surrogate mean {surrogate.mean():.4f}  max {surrogate.max():.4f}")


def file_keys(nd):
    """Columns that identify which file a row came from."""
    return {'group': f"{nd['dmd']}_{nd['channel']}", 'dmd': str(nd['dmd']),
            'channel': str(nd['channel']), 'phase': str(nd['phase'])}


idx.drop(columns='path')

## 3. Running the check

The ephys notebook has a `validate/` run — one sweep point, no surrogates — compared
with a single rawresults file. There is none for SLAP2, but the same idea works on
the full grid because **the null's grid is the pipeline's grid**:
`family_max[:, :, 0].max(axis=1)` is the maximum over every filter *and* every one of
the 18 (delay, duration) cells, which is exactly what the optimized results reduce
their `*_all_values.csv` to per unit. `optimize_waven_parameters` writes that csv
beside every optimized result — one row per (ROI, delay, duration).

The check is run for all four (DMD, channel) files of the chosen phase. The **signal
is part of the address**: dF/F and events results share every file name, and the
comparison at the bottom shows what comparing against the wrong one looks like.

In [ ]:
def all_values(dmd, channel, phase, session=SESSION, opt_dir=OPT_DIR):
    """The per-(unit, delay, duration) abs_max_value table for one result set."""
    stem = f'optimized__{session}__{dmd}__{channel}__phase_{phase}_all_values.csv'
    return pd.read_csv(opt_dir / stem).astype({'unit_id': str})


def check_file(dmd, channel, phase, opt_dir=OPT_DIR):
    """Null lag 0 vs the optimized results' whole-grid maximum, per unit of one file."""
    nd = load_null(pick(idx, session=SESSION, signal=SIGNAL, dmd=dmd, channel=channel,
                        phase=str(phase)))
    av = all_values(dmd, channel, phase, opt_dir=opt_dir)
    # the rule of the ephys notebook's section 8: same grid, or the comparison is void
    assert set(np.round(av.delay, 4)) == set(np.round(nd['sweep_delays'], 4))
    assert set(np.round(av.duration, 4)) == set(np.round(nd['sweep_durations'], 4))
    raw = av.groupby('unit_id')['abs_max_value'].max()
    lag0 = pd.Series(nd['family_max'][:, :, 0].max(axis=1), index=nd['unit_ids'])
    out = pd.concat([lag0.rename('null_lag0'), raw.rename('rawresults')], axis=1, join='inner')
    out['group'] = f'{dmd}_{channel}'
    out.attrs['n_raw'] = len(raw)
    return out


files = idx[(idx.session == SESSION) & (idx.signal == SIGNAL) & (idx.phase == PHASE)]
checks = [check_file(r.dmd, r.channel, PHASE) for r in files.itertuples()]
both = pd.concat(checks)
diff = (both.null_lag0 - both.rawresults).abs()

summary = pd.DataFrame({
    'units in null': [len(c) for c in checks],
    'units in results': [c.attrs['n_raw'] for c in checks],
    'max |diff|': [(c.null_lag0 - c.rawresults).abs().max() for c in checks],
    'median |diff|': [(c.null_lag0 - c.rawresults).abs().median() for c in checks],
}, index=[c.group.iloc[0] for c in checks])
print(f'{len(both)} units over {len(checks)} files (phase {PHASE}, signal {SIGNAL})')
print(f'max  abs diff    {diff.max():.2e}')
print(f'median abs diff  {diff.median():.2e}')
print(f'relative         {(diff / both.rawresults).max():.2e} worst case\n')
summary.round(7)

The residual is `~1e-5` against a tolerance of `~1e-4`, limited by the float16
stimulus cache as in ephys. Every ROI of the results is present in the null here (250 of 250), so the `min_active_samples` screen removed nothing from this session. The null also
reproduces the pipeline's **coverage rule** — the frames dropped for lack of imaging
coverage depend on both delay and duration, which is why the null has 18 distinct row
selections — and that is part of what this comparison confirms.

The same comparison against the *events* results, which are the wrong signal for
this null:

In [ ]:
wrong = pd.concat([check_file(r.dmd, r.channel, PHASE, opt_dir=OPT_ROOT / SESSION / 'events')
                   for r in files.itertuples()])
d_wrong = (wrong.null_lag0 - wrong.rawresults).abs()
print(f'against the events results: max |diff| {d_wrong.max():.2e}, '
      f'median {d_wrong.median():.2e}   (dF/F: {diff.max():.1e})')
print('every unit lines up by id and the plot would look plausible - the number is the only tell.')

In [ ]:
fig, ax = plt.subplots(figsize=(4.4, 4.4))
ax.plot(both.rawresults, both.null_lag0, '.', ms=3, alpha=.5)
lims = [both[['rawresults', 'null_lag0']].values.min() * .98,
        both[['rawresults', 'null_lag0']].values.max() * 1.02]
ax.plot(lims, lims, 'k--', lw=.8, label='identity')
ax.set(xlabel='optimized results: max over the grid', ylabel='null file, lag 0',
       title=f'{len(both)} units\nmax |diff| = {diff.max():.1e}', xlim=lims, ylim=lims)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. The per-unit statistic, and a null that does not stop at 1/(1+K)

The statistic is one number per unit: the largest |correlation| it reaches anywhere
in the search — over every filter, every family and every one of the 18 sweep points.
`family_max[u, :, 0].max()` is the observed value and
`family_max[u, :, 1:].max(0)` are the `K` surrogate values of the same quantity.

**Counting is the obvious null, and it has a floor.** With `K = 300` the smallest
p-value expressible is `1/(1+K) = 0.0033`. BH compares the k-th smallest p against
`q·k/n`, so counting can only reject once the BH line has climbed to the floor — at
rank `n·floor/q`. With 250 ROIs that is rank ~17, well within reach of the units that
respond. **Unlike ephys (K = 100, floor 0.0099, ~3 100 units), counting is not
hopeless here** — the question this chapter asks is what the fitted tail adds on top
of what counting already gives.

**Fit the tail as well as counting it.** The statistic is a maximum over a very large
number of weakly dependent quantities, so by Fisher–Tippett it converges to an
extreme-value law — the **Gumbel** for light-tailed summands. The `K` surrogates are
`K` draws from that null, so fitting a Gumbel and reading the observed value off the
fitted tail gives a continuous p-value, unbounded below (method of moments):

$$\beta = s\sqrt{6}/\pi, \qquad \mu = \bar{x} - \gamma\beta, \qquad
p = 1 - \exp\!\big(-e^{-(x-\mu)/\beta}\big)$$

The fit is **per unit**: the null scale depends on the unit's own trace, so one pooled
fit would be wrong for most. The extrapolation below `1/(1+K)` is not measured; two
checks keep it honest, both run below.

In [ ]:
EULER = 0.5772156649015329                 # Euler-Mascheroni, for the Gumbel mean


def gumbel_mom(draws):
    """Method-of-moments Gumbel (mu, beta), one fit per ROW of `draws`."""
    beta = draws.std(axis=1, ddof=1) * np.sqrt(6) / np.pi
    return draws.mean(axis=1) - EULER * beta, beta


def gumbel_sf(x, mu, beta):
    """P(X > x) under Gumbel(mu, beta); expm1 keeps its digits when p is tiny."""
    return -np.expm1(-np.exp(-(x - mu) / beta))


def bh_fdr(p):
    """Benjamini-Hochberg adjusted p-values (q-values). See section 6."""
    p = np.asarray(p, dtype=float)
    n = len(p)
    order = np.argsort(p)
    q = np.minimum.accumulate((n / np.arange(n, 0, -1)) * p[order][::-1])[::-1]
    out = np.empty(n)
    out[order] = np.minimum(q, 1.0)
    return out


def unit_stats(path):
    """Per-unit statistic, null and p-values for one file.

    Returns (df, M_sur). `M_sur` is (n_units, K): each unit's K surrogate
    sweep-wide maxima - the histogram section 5 plots, and the sample every fit
    here is made from.
    """
    nd = load_null(path)
    fm = nd['family_max']
    obs, sur = fm[:, :, 0], fm[:, :, 1:]
    K = sur.shape[2]
    if K == 0:
        raise ValueError(f'{nd["path"]} holds no surrogates.')

    M_obs, M_sur = obs.max(axis=1), sur.max(axis=1)        # the statistic
    mu, beta = gumbel_mom(M_sur)

    # studentise per family first, then maximise - the variant compared below
    m, s = sur.mean(axis=2), sur.std(axis=2, ddof=1)
    T_obs = ((obs - m) / s).max(axis=1)
    T_sur = ((sur - m[:, :, None]) / s[:, :, None]).max(axis=1)
    t_mu, t_beta = gumbel_mom(T_sur)

    df = pd.DataFrame({
        **file_keys(nd),
        'unit_id': nd['unit_ids'],
        'n_active': nd['n_spikes'],
        'max_obs': M_obs,
        'null_mean': M_sur.mean(axis=1),
        'null_p95': np.quantile(M_sur, 0.95, axis=1),
        'mu': mu,
        'beta': beta,
        'p_count': (1 + (M_sur >= M_obs[:, None]).sum(axis=1)) / (1 + K),
        'p': gumbel_sf(M_obs, mu, beta),                   # the one used throughout
        'p_stud': gumbel_sf(T_obs, t_mu, t_beta),
    })
    return df, M_sur


def session_stats(phase=PHASE, session=SESSION, signal=SIGNAL, q_level=Q_LEVEL):
    """Every (DMD, channel) file of one phase, pooled into a single table.

    BH is applied HERE, over all files at once - the family is the session, and
    that choice is what chapter 6 is about.
    """
    files = idx[(idx.session == session) & (idx.signal == signal) & (idx.phase == str(phase))]
    parts = [unit_stats(r.path) for r in files.itertuples()]
    df = pd.concat([d for d, _ in parts], ignore_index=True)
    M = np.vstack([m for _, m in parts])
    df['q'] = bh_fdr(df.p)
    df['significant'] = df.q <= q_level
    df.attrs.update(K=int(M.shape[1]), phase=phase, q_level=q_level, n_groups=len(parts))
    return df, M


session, M_sur = session_stats()
K = session.attrs['K']
FLOOR = 1 / (1 + K)
print(f'pooled: {len(session)} ROIs from {session.attrs["n_groups"]} (DMD, channel) files, '
      f'phase {session.attrs["phase"]}, K={K}')
print(f'counting floor 1/(1+K) = {FLOOR:.4f};  '
      f'{int((session.p_count <= FLOOR).sum())} units are pinned there')
print(f'fitted p reaches {session.p.min():.1e}')
assert session.unit_id.is_unique          # ids carry the DMD and channel, unlike the mesoscope
session.head(4).round(4)

### Does the fit deserve the trust?

**Does it agree with counting where counting works?** Restrict to units whose counted
p is neither at the floor nor uselessly large, and compare — with 250 units the
sample is small, so read the correlation loosely.

**Is the statistic really Gumbel?** Each unit's own `K` draws are the sample, so the
Q-Q residual against its own fit says how well the law holds, and the sign of the
residual in the *upper* tail says which way the extrapolation errs.

Studentising each family before maximising is a different, more powerful statistic;
`p` holds the raw maximum and `p_stud` is carried alongside so the difference stays
visible.

**Reading the numbers.** The fit tracks counting closely where counting resolves. The
upper-tail bias is *positive* — the top order statistics sit slightly above the fitted
Gumbel, so the fit is if anything mildly optimistic at the far tail, the opposite sign
to the conservative fit in the ephys notebook. The marginal units sit near p ~ 1e-2
where counting still resolves, so this changes little; treat very small fitted
p-values as "far beyond the surrogates", not as calibrated probabilities.

In [ ]:
def fit_quality(M):
    """Per-unit Q-Q residual against its own fit, in units of the fitted beta.

    Scale-free, so statistics with different units can be compared. The second
    return is the signed residual over the top 10 order statistics: negative
    means the draws are LIGHTER than the fit there, i.e. the fitted tail is too
    heavy and the extrapolated p is conservative.
    """
    K = M.shape[1]
    mu, beta = gumbel_mom(M)
    plotting = (np.arange(1, K + 1) - 0.5) / K
    q_theory = mu[:, None] - beta[:, None] * np.log(-np.log(plotting))[None, :]
    resid = np.sort(M, axis=1) - q_theory
    return (np.sqrt((resid ** 2).mean(axis=1)) / beta,
            resid[:, -10:].mean(axis=1) / beta)


mid = session[(session.p_count > 0.02) & (session.p_count < 0.5)]
rmse, tail = fit_quality(M_sur)
print(f'AGREEMENT with counting, on the {len(mid)} units where counting still resolves:')
print(f'  spearman(p_count, p)      {mid.p_count.corr(mid.p, method="spearman"):.3f}')
print(f'  median p / p_count        {float((mid.p / mid.p_count).median()):.2f}')
print(f'\nFIT QUALITY (residual in units of the fitted beta, over {K} draws each):')
print(f'  median Q-Q rmse           {np.median(rmse):.3f}   (90th pct {np.quantile(rmse, .9):.3f})')
print(f'  median upper-tail bias    {np.median(tail):+.3f}   '
      f'({"fit heavier than draws -> conservative" if np.median(tail) < 0 else "unbiased / optimistic"})')

print(f'\nRAW vs STUDENTISED, same units, same draws:')
for name, col in (('raw max |r|  (p)', 'p'), ('studentised (p_stud)', 'p_stud')):
    n_rej = int((bh_fdr(session[col]) <= Q_LEVEL).sum())
    print(f'  {name:22s} BH rejections {n_rej:4d}')
print(f'  rank agreement between them   '
      f'spearman {session.p.corr(session.p_stud, method="spearman"):.3f}')

## 5. Individual units in detail

Units drawn at random from the pooled session (seed `SEED`), plus one that is not
random. Each panel shows the whole computation for one ROI:

* the **histogram** of its `K` surrogate sweep-wide maxima — the null,
* the **Gumbel** fitted to exactly those draws,
* the **observed** maximum,
* and the numbers that come out: the counted p, the fitted p and the session-wide q
  from chapter 6.

A random draw is mostly null. The panel axis autoscales to include the observed line,
so in a null panel the line sits inside the grey histogram, while a responsive ROI is
many `beta` beyond its surrogates and its null is squeezed into a sliver at the left
edge. Titles carry the (DMD, channel) file: red-channel ROIs are expected to look
null far more often than green ones, for the kinetics reason above.

In [ ]:
rng = np.random.default_rng(SEED)
picks = np.sort(rng.choice(len(session), 11, replace=False))
# plus one that is NOT random: the strongest unit in the session, so the figure
# also shows what the other case looks like
strongest = int(session.p.values.argmin())
panels = list(picks) + [strongest]

n_panels = len(panels)
ncols = min(6, n_panels)
nrows = int(np.ceil(n_panels / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 3.6 * nrows),
                         squeeze=False)
flat = axes.flatten()
for ax, i in zip(flat, panels):
    row = session.iloc[i]
    draws = M_sur[i]

    ax.hist(draws, bins=16, density=True, color='lightgrey', edgecolor='grey',
            label=f'{len(draws)} surrogates')
    spread = draws.max() - draws.min()
    x = np.linspace(draws.min() - 0.2 * spread, draws.max() + 0.35 * spread, 400)
    z = (x - row.mu) / row.beta
    ax.plot(x, np.exp(-(z + np.exp(-z))) / row.beta, color='navy', lw=1.6,
            label='Gumbel fit')
    # the axis autoscales to include this line: a responsive unit squeezes its own
    # histogram into a sliver at the left edge, which is the point of the panel
    ax.axvline(row.max_obs, color='crimson', lw=2, label='observed')

    tag = 'strongest in session (not random)' if i == strongest else f'{row.n_active} {ACTIVITY}'
    ax.set_title(f'{row.group}  {str(row.unit_id)[:12]}\n{tag}', fontsize=9,
                 color='crimson' if i == strongest else 'black')
    ax.set_xlabel('sweep-wide max |r|', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.text(0.97, 0.97,
            f'observed  {row.max_obs:.3f}\n'
            f'p count   {row.p_count:.3f}\n'
            f'p Gumbel  {row.p:.1e}\n'
            f'q         {row.q:.3f}\n'
            f'{"SIGNIFICANT" if row.significant else "n.s."}',
            transform=ax.transAxes, ha='right', va='top', fontsize=7,
            family='monospace',
            bbox=dict(facecolor='w', alpha=.85, linewidth=0, pad=2))
for ax in flat[n_panels:]:              # unused cells of the last row
    ax.axis('off')
for r in range(nrows):
    axes[r, 0].set_ylabel('density')
flat[0].legend(fontsize=7, loc='center right')
plt.tight_layout()
plt.show()

print(f'{len(picks)} drawn with seed {SEED}: rows {[int(i) for i in picks]} '
      f'of {len(session)};\nplus row {strongest}, the strongest unit in the session')
print(session.iloc[panels][['group', 'unit_id', 'n_active', 'max_obs', 'null_mean',
                            'p_count', 'p', 'q', 'significant']].to_string(index=False))
n_sig_drawn = int(session.iloc[picks].significant.sum())
print(f'\nA fair draw: {100 * (1 - session.significant.mean()):.0f} % of this session is '
      f'not significant, and {n_sig_drawn} of the {len(picks)} random units came out '
      f'significant.')
s = session.iloc[strongest]
print(f'The last panel: observed {s.max_obs:.3f} is {s.max_obs / s.null_mean:.1f}x its null '
      f'mean and {(s.max_obs - s.mu) / s.beta:.0f} fitted beta past mu.')

## 6. FDR across the whole session

The per-unit p-value already accounts for everything *inside* a unit: it is the null of
a maximum over every filter and every sweep point, so no correction over filters is
owed. What it does not cover is the 250 ROIs. At `p ≤ 0.05` about one in twenty passes
by chance, so a correction across units goes on top, and **Benjamini–Hochberg** is the
one to use: it controls the expected fraction of false claims among those declared
significant.

The family here is **the whole session** — every ROI on both DMDs and both channels of
one phase, pooled. That is a choice: the red channel responds barely at all, and
pooling it in with the green raises the bar slightly for the green units. Correcting
per channel would be more permissive, and the family must never be chosen after seeing
the p-values. The two phases are separate tests: keeping the better of two is two
tests, not one.

BH rejects the `k` smallest p-values where `p(k) ≤ q·k/n`. With the counted p this is
possible here, unlike in ecephys, because the floor `1/(1+K)` is below the BH line at
rank `n·floor/q` and enough units sit at it. What the fitted tail adds is
**resolution above the floor**: units whose counted p is a handful of exceedances out of 300, which counting ranks in coarse steps of 0.0033. Those are also the marginal units, so their status depends on the tail shape of the fit — report both numbers.

In [ ]:
def bh_rejections(p, q_level=Q_LEVEL):
    """How many hypotheses BH rejects, without computing every q-value."""
    p = np.sort(np.asarray(p, dtype=float))
    n = len(p)
    below = np.where(p <= q_level * np.arange(1, n + 1) / n)[0]
    return (below.max() + 1) if len(below) else 0


n = len(session)
R_fit, R_cnt = bh_rejections(session.p), bh_rejections(session.p_count)
rank_at_floor = int(np.ceil(n * FLOOR / Q_LEVEL))
print(f'family = the whole session: {n} units, q = {Q_LEVEL}\n')
print(f'  BH on the counted p   {R_cnt:4d} rejections'
      f'   (floor {FLOOR:.4f}; the BH line reaches it at rank {rank_at_floor}, '
      f'{int((session.p_count <= FLOOR).sum())} units are pinned there)')
print(f'  BH on the fitted p    {R_fit:4d} rejections')
print(f'  units at p <= 0.05    {int((session.p <= 0.05).sum()):4d}   (uncorrected)')

fit_only = session[session.significant & (bh_fdr(session.p_count) > Q_LEVEL)]
print(f'\nthe fit adds {len(fit_only)} units the counted p does not reject; their counted p '
      f'lies in [{fit_only.p_count.min():.4f}, {fit_only.p_count.max():.4f}]'
      if len(fit_only) else '\nthe fit adds no units beyond the counted p')

by_group = (session.groupby('group')
                   .agg(units=('unit_id', 'size'),
                        p05=('p', lambda s: int((s <= 0.05).sum())),
                        significant=('significant', 'sum'),
                        median_obs=('max_obs', 'median'),
                        median_null=('null_mean', 'median'))
                   .assign(pct=lambda d: (100 * d.significant / d.units).round(1)))
print('\nwhere the session-wide rejections land:')
by_group

### The correction in one picture

BH sorts the p-values and compares the k-th smallest against the line `q·k/n`. It
rejects everything up to the **last** rank that still falls below the line — the
step-up property. With ephys' K = 100 the grey counted curve never came down to the
red line; with K = 300 and only 250 units it does, and the fitted curve differs from it
only in the units just beyond the shelf at the floor. At n = 250 the whole difference
between the two is a handful of units, so the picture is mainly a check that counting
and fitting tell the same story.

In [ ]:
k = np.arange(1, n + 1)
bh_line = Q_LEVEL * k / n
p_fit = np.sort(session.p.values)
p_cnt = np.sort(session.p_count.values)
p_star = Q_LEVEL * R_fit / n
CLIP = 1e-12                       # the deepest fitted p can be far below this; clip for display

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

# --- full range, log-log ---
ax1.loglog(k, np.maximum(p_fit, CLIP), lw=1.6, color='navy', label='fitted p (Gumbel)')
ax1.loglog(k, p_cnt, lw=1.6, color='0.6', label=f'counted p (K={K})')
ax1.loglog(k, bh_line, 'r-', lw=1.2, label=f'BH line  q·k/n  (q={Q_LEVEL})')
ax1.axhline(FLOOR, color='navy', ls=':', lw=1, label=f'counting floor {FLOOR:.4f}')
ax1.axvline(R_fit, color='navy', ls='--', lw=1)
ax1.axvline(R_cnt, color='0.5', ls='--', lw=1)
ax1.set(xlabel='rank k', ylabel='p-value', ylim=(CLIP, 1.4),
        title=f'all {n} units, pooled over {session.attrs["n_groups"]} {GROUPS}')
ax1.legend(fontsize=8, loc='upper left')

# --- zoom on the crossing, linear ---
zoom = int(1.6 * max(R_fit, R_cnt))
ymax = 1.4 * max(bh_line[zoom - 1], FLOOR)
ax2.plot(k[:zoom], p_fit[:zoom], lw=2, color='navy', label=f'fitted p  (R = {R_fit})')
ax2.plot(k[:zoom], p_cnt[:zoom], lw=2, color='0.6', label=f'counted p  (R = {R_cnt})')
ax2.plot(k[:zoom], bh_line[:zoom], 'r-', lw=1.4, label='BH line')
ax2.axhline(FLOOR, color='navy', ls=':', lw=1)
ax2.axvline(R_fit, color='navy', ls='--', lw=1)
ax2.axvline(R_cnt, color='0.5', ls='--', lw=1)
ax2.plot([R_fit], [p_star], 'ko', ms=6, zorder=5)
ax2.set(xlabel='rank k', ylabel='p-value', xlim=(0, zoom), ylim=(0, ymax),
        title='zoom: the last crossing of each flavour')
ax2.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

print(f'BH rejects the {R_fit} smallest fitted p-values (p* = {p_star:.2e}) and the '
      f'{R_cnt} smallest counted ones.')
print(f'The counted p-values sit at {FLOOR:.4f} for their first '
      f'{int((p_cnt <= FLOOR).sum())} ranks; the BH line only reaches that height at '
      f'rank {rank_at_floor}.')

### Anatomy

There is no CCF annotation for SLAP2 ROIs, so the breakdown above by DMD and channel is
the closest analogue of ephys' by-structure table — and the split that matters is the
channel. Green (iGluSnFR4f) carries essentially all of the rejections; red (RCaMP3) is
near zero on both DMDs, with a median peak correlation no higher than its own null mean.

In [ ]:
by_channel = (session.groupby('channel')
                     .agg(units=('unit_id', 'size'), significant=('significant', 'sum'),
                          median_obs=('max_obs', 'median'), median_null=('null_mean', 'median'))
                     .assign(pct=lambda d: (100 * d.significant / d.units).round(1)))
print(f'{int(session.significant.sum())} session-wide rejections, by channel:')
print(by_channel.to_string())

### The same data without the correction

The reading that skips it: each unit has 78 families, and asking "did the observed
value beat *any* family's own 95th percentile" is 78 chances to be wrong.

In [ ]:
WALK_MASK = (session.dmd == WALK["dmd"]) & (session.channel == WALK["channel"])
WALK_NAME = f'{WALK["dmd"]} {WALK["channel"]}'

# `observed` and `surrogate` are the walk file from section 2
crit = np.quantile(surrogate, 0.95, axis=2)              # (units, families)
naive = (observed > crit).any(axis=1)
walk_rows = session[WALK_MASK]

print(f'{WALK_NAME}, the same {len(walk_rows)} units three ways:')
print(f'  beats its own 95th pct in ANY of {n_families} families   '
      f'{int(naive.sum()):4d}  ({100 * naive.mean():.0f} %)')
print(f'  p <= 0.05 on the sweep-wide maximum                  '
      f'{int((walk_rows.p <= 0.05).sum()):4d}  '
      f'({100 * (walk_rows.p <= 0.05).mean():.0f} %)')
print(f'  survives BH across the whole session                 '
      f'{int(walk_rows.significant.sum()):4d}  '
      f'({100 * walk_rows.significant.mean():.0f} %)')
print('\nThe first number is why family_max keeps the family axis instead of storing')
print('one value per unit: the maximum has to be taken over families BEFORE the')
print('comparison, not after it.')

### Peak correlation by (DMD, channel): significant vs. all other ROIs

The box-per-group plot of `demo_best_ephys_gabor_rfs`, with two boxes per group: the
ROIs that survive the session-wide correction and every other ROI in the recording.
The statistic is `max_obs`, the value the test itself used.

In [ ]:
XLABEL = "DMD / channel"

import seaborn as sns

box = session.assign(group_label=session.group)
hue_order = [f'significant (BH q <= {Q_LEVEL})', 'all other units']
box['status'] = np.where(box.significant, hue_order[0], hue_order[1])

n_sig = box[box.significant].group.value_counts()
n_all = box.group.value_counts()
order = sorted(n_all.index)
labels = [f'{g} ({n_sig.get(g, 0)}/{n_all[g] - n_sig.get(g, 0)})' for g in order]

fig, ax = plt.subplots(figsize=(max(8, 1.4 * len(order)), 5))
sns.boxplot(data=box, x='group', y='max_obs', hue='status', order=order,
            hue_order=hue_order, palette=['crimson', '0.7'], fliersize=2, ax=ax)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=11)
ax.set_xlabel(f'{XLABEL} (n significant / n other)', fontsize=13)
ax.set_ylabel('max(|Corr|)', fontsize=13)
ax.legend(fontsize=11, loc='upper right')
ax.grid()
fig.suptitle(f'Peak correlation by {XLABEL}: significant vs. other units '
             f'(n={int(box.significant.sum())} / {int((~box.significant).sum())})',
             fontsize=15)
fig.tight_layout()
plt.show()